# Run NMME SST-index preprocessing

This notebook generates NMME SST/ELI index time series used by `4_refactor_sst_skill_ts.ipynb`.

It calls `scripts/run_nmme_nino34_yeager_diag.py`, which reads the member-split NMME SST archive and writes drift-corrected monthly/seasonal index time series. It does not compute skill; SST-index skill is computed later in `4_refactor_sst_skill_ts.ipynb` using the HadISST2 indices generated by `0_run_sigmod_sst_index.ipynb`.

Primary output:

`/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries/NMME<MM>_<REGION>SST_<mon|seas>_dd_<DATA_START>_<DATA_END>.nc`


In [1]:
import os
import re
import subprocess
import json
import sys
from pathlib import Path

import xarray as xr

# Identify repository root. This works when the notebook is run from either
# the repository root or the jupyter/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent

SCRIPT_PATH = REPO_ROOT / "scripts" / "run_nmme_nino34_yeager_diag.py"
print(f"Repository root: {REPO_ROOT}")
print(f"Python         : {sys.executable}")
print(f"Script         : {SCRIPT_PATH}")


Repository root: /global/u2/z/zhan391/code/ESP-Lab
Python         : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
Script         : /global/u2/z/zhan391/code/ESP-Lab/scripts/run_nmme_nino34_yeager_diag.py


## Configuration

Set `MODELS` to an explicit list of NMME model directory names when you want to process only selected models. Leave `MODELS = []` to use `MODEL_SET = "all"` for every downloaded SST model or `MODEL_SET = "yeager-f03"` for the smaller eight-model Yeager-style subset. Set `FORCE = True` only when you want to rebuild cached per-model regional SST-index anomaly files.


In [2]:
NMME_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member")
OUTDIR = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME")
NMME_FIXED_DIR = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/fixed")
MODEL_SET = "all"  # Used only when MODELS is empty: "all" or "yeager-f03".

REGIONS = [
    "IOD", "TNI", "ONI", "RONI",
    "Nino12", "Nino3", "Nino3.4", "Nino4",
    "TNA", "TSA", "PACWRAMPOOL", "AtlNino", "AtlMDR", "ELI"
]  # List of target indices/regions to compute
CUSTOM_REGIONS = {
    "Nino12": {
        "lonlat": [270.0, 280.0, -10.0, 0.0],
        "long_name": "Nino 1+2 regional mean SST",
    },
    "Nino3": {
        "lonlat": [210.0, 270.0, -5.0, 5.0],
        "long_name": "Nino 3 regional mean SST",
    },
    "Nino3.4": {
        "lonlat": [190.0, 240.0, -5.0, 5.0],
        "long_name": "Nino 3.4 regional mean SST",
    },
    "Nino4": {
        "lonlat": [160.0, 210.0, -5.0, 5.0],
        "long_name": "Nino 4 regional mean SST",
    },
    "TNA": {
        "lonlat": [305.0, 345.0, 5.0, 25.0],
        "long_name": "TNA regional mean SST",
    },
    "TSA": {
        "lonlat": [330.0, 10.0, -20.0, 0.0],
        "long_name": "TSA regional mean SST",
    },
    "PACWRAMPOOL": {
        "lonlat": [60.0, 170.0, -15.0, 15.0],
        "long_name": "PACWRAMPOOL regional mean SST",
    },
    "AtlNino": {
        "lonlat": [340.0, 360.0, -3.0, 3.0],
        "long_name": "Atlantic Nino regional mean SST",
    },
    "AtlMDR": {
        "lonlat": [280.0, 350.0, 10.0, 20.0],
        "long_name": "Atlantic MDR regional mean SST",
    },
    # Helper regions for derived indices
    "IOD_West": {
        "lonlat": [50.0, 70.0, -10.0, 10.0],
        "long_name": "IOD West regional mean SST",
    },
    "IOD_East": {
        "lonlat": [90.0, 110.0, -10.0, 0.0],
        "long_name": "IOD East regional mean SST",
    },
    "TropicalMean": {
        "lonlat": [0.0, 360.0, -20.0, 20.0],
        "long_name": "Tropical Mean regional mean SST",
    },
}


# Explicit model names to process. Use [] to fall back to MODEL_SET.
MODELS = [
    "CanSIPS-IC3",
    "GFDL-CM2p1",
    "NASA-GMAO-062012",
    "CanSIPS-IC4",
    "GFDL-CM2p1-aer04",
    "NCAR-CESM1",
    "CanSIPSv2",
    "GFDL-CM2p5-FLOR-A06",
    "NCEP-CFSv1",
    "CMC1-CanCM3",
    "GFDL-CM2p5-FLOR-B01",
    "NCEP-CFSv2",
    "CMC2-CanCM4",
    "GFDL-SPEAR",
    "GEM-NEMO",
    "NASA-GMAO",
    "CanCM4i",
    "NASA-GEOSS2S",
    "COLA-RSMAS-CCSM3",
    "IRI-ECHAM4p5-AnomalyCoupled",
    "COLA-RSMAS-CCSM4",
    "IRI-ECHAM4p5-DirectCoupled",
    "COLA-RSMAS-CESM1",
]
# "standard" retains models spanning 1981-2010. "common" uses the
# all-model overlap for both processing and climatology (currently 1991-2009).
PERIOD_MODE = "standard"
DATA_START = 1980  # Use "auto" for earliest selected-model year, or set an integer year.
DATA_END = 2020    # Use "auto" for latest selected-model year, or set an integer year.
CLIM_START = 1981
CLIM_END = 2010
NMME_SST_LAND_MASK = True
FORCE = False

YEAGER_F03_MODELS = [
    "CMC1-CanCM3",
    "CMC2-CanCM4",
    "COLA-RSMAS-CCSM4",
    "GFDL-CM2p1-aer04",
    "GFDL-CM2p5-FLOR-A06",
    "GFDL-CM2p5-FLOR-B01",
    "NASA-GMAO-062012",
    "NCEP-CFSv2",
]


## Validate inputs

In [3]:
missing = []
for label, path in {
    "script": SCRIPT_PATH,
    "NMME root": NMME_ROOT,
}.items():
    if not path.exists():
        missing.append(f"{label}: {path}")

if MODEL_SET not in {"all", "yeager-f03"}:
    missing.append(f"MODEL_SET must be 'all' or 'yeager-f03', got {MODEL_SET!r}")

if missing:
    raise FileNotFoundError("Missing or invalid configuration:\n" + "\n".join(missing))

available_models = sorted(
    p.name for p in NMME_ROOT.iterdir()
    if p.is_dir() and p.name != "logs" and (p / "sst").is_dir()
)

EXPLICIT_MODELS = list(
    dict.fromkeys(str(model).strip() for model in MODELS if str(model).strip())
)
selection_errors = []
if EXPLICIT_MODELS:
    unavailable_models = sorted(set(EXPLICIT_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODELS contains names that are not available under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = EXPLICIT_MODELS
    model_selection = "explicit MODELS list"
elif MODEL_SET == "all":
    SELECTED_MODELS = available_models
    model_selection = "MODEL_SET='all'"
else:
    unavailable_models = sorted(set(YEAGER_F03_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODEL_SET='yeager-f03' includes unavailable models under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = YEAGER_F03_MODELS
    model_selection = "MODEL_SET='yeager-f03'"

if selection_errors:
    raise ValueError("Invalid model selection:\n" + "\n".join(selection_errors))

s_chunk_pattern = re.compile(r"_S(\d+)-(\d+)\.nc$")

def _chunk_range(path):
    match = s_chunk_pattern.search(path.name)
    return (int(match.group(1)), int(match.group(2))) if match else None

def _decode_s_edge(path, first=True):
    with xr.open_dataset(path, decode_times=False) as ds:
        s_ds = ds[["S"]].copy()
    if s_ds["S"].attrs.get("calendar") == "360":
        s_ds["S"].attrs["calendar"] = "360_day"
    decoded = xr.decode_cf(s_ds, decode_times=True)["S"]
    return decoded.values[0 if first else -1]

def _model_s_years(model):
    files = []
    for path in (NMME_ROOT / model / "sst").glob("M*/*.nc"):
        chunk_range = _chunk_range(path)
        if chunk_range is not None:
            files.append((*chunk_range, path))
    if not files:
        raise FileNotFoundError(f"No SST chunks found for {model}")
    first_file = min(files, key=lambda item: (item[0], item[1]))[2]
    last_file = max(files, key=lambda item: (item[1], item[0]))[2]
    return _decode_s_edge(first_file, first=True).year, _decode_s_edge(last_file, first=False).year

candidate_model_year_ranges = {model: _model_s_years(model) for model in SELECTED_MODELS}
if PERIOD_MODE not in {"standard", "common"}:
    raise ValueError(f"PERIOD_MODE must be 'standard' or 'common', got {PERIOD_MODE!r}")
candidate_common_start = max(start for start, _ in candidate_model_year_ranges.values())
candidate_common_end = min(end for _, end in candidate_model_year_ranges.values())
if candidate_common_start > candidate_common_end:
    raise ValueError("Candidate NMME models have no overlapping year window.")
if PERIOD_MODE == "common":
    CLIM_START = candidate_common_start
    CLIM_END = candidate_common_end
    DATA_START = candidate_common_start
    DATA_END = candidate_common_end

# Keep only models that contain every year in the standard 30-year normal.
excluded_for_climatology = {
    model: (first_year, last_year)
    for model, (first_year, last_year) in candidate_model_year_ranges.items()
    if first_year > CLIM_START or last_year < CLIM_END
}
SELECTED_MODELS = [
    model for model in SELECTED_MODELS
    if model not in excluded_for_climatology
]
if not SELECTED_MODELS:
    raise ValueError(
        f"No selected NMME model spans the full {CLIM_START}-{CLIM_END} climatology."
    )
model_year_ranges = {
    model: candidate_model_year_ranges[model]
    for model in SELECTED_MODELS
}
DATA_START_YEAR = min(start for start, _ in model_year_ranges.values()) if DATA_START == "auto" else int(DATA_START)
DATA_END_YEAR = max(end for _, end in model_year_ranges.values()) if DATA_END == "auto" else int(DATA_END)
if DATA_START_YEAR > DATA_END_YEAR:
    raise ValueError(f"DATA_START ({DATA_START_YEAR}) must be <= DATA_END ({DATA_END_YEAR})")

def _region_token(region):
    return str(region).replace(".", "")


def _index_file_label(region):
    token = _region_token(region)
    return "ELI" if region == "ELI" else f"{token}SST"


EXPECTED_TIMESERIES_FILES = {
    (region, init_month, freq): OUTDIR / "sst_index" / "timeseries" / f"NMME{init_month:02d}_{_index_file_label(region)}_{freq}_dd_{DATA_START_YEAR}_{DATA_END_YEAR}.nc"
    for region in REGIONS
    for init_month in [2, 5, 8, 11]
    for freq in ["mon", "seas"]
}
EXPECTED_NINO34_TIMESERIES_FILE = EXPECTED_TIMESERIES_FILES[("Nino3.4", 5, "seas")]

print(f"Period mode: {PERIOD_MODE}")
print(f"Downloaded NMME SST model directories: {len(available_models)}")
print(available_models)
print(f"\nCandidate models from {model_selection}: {len(candidate_model_year_ranges)}")
print(f"Eligible models spanning {CLIM_START}-{CLIM_END}: {len(SELECTED_MODELS)}")
print(SELECTED_MODELS)
if excluded_for_climatology:
    print("\nExcluded models without the complete climatology window:")
    for model, (first_year, last_year) in excluded_for_climatology.items():
        print(f"  {model}: available {first_year}-{last_year}")
print(f"\nSelected-model raw data window: {DATA_START_YEAR}-{DATA_END_YEAR}")
print(f"Drift climatology window: {CLIM_START}-{CLIM_END}")
print(f"Expected time-series files: {len(EXPECTED_TIMESERIES_FILES)}")
print(f"Nino3.4 example time-series file: {EXPECTED_NINO34_TIMESERIES_FILE}")


Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)


Period mode: standard
Downloaded NMME SST model directories: 23
['CMC1-CanCM3', 'CMC2-CanCM4', 'COLA-RSMAS-CCSM3', 'COLA-RSMAS-CCSM4', 'COLA-RSMAS-CESM1', 'CanCM4i', 'CanSIPS-IC3', 'CanSIPS-IC4', 'CanSIPSv2', 'GEM-NEMO', 'GFDL-CM2p1', 'GFDL-CM2p1-aer04', 'GFDL-CM2p5-FLOR-A06', 'GFDL-CM2p5-FLOR-B01', 'GFDL-SPEAR', 'IRI-ECHAM4p5-AnomalyCoupled', 'IRI-ECHAM4p5-DirectCoupled', 'NASA-GEOSS2S', 'NASA-GMAO', 'NASA-GMAO-062012', 'NCAR-CESM1', 'NCEP-CFSv1', 'NCEP-CFSv2']

Candidate models from explicit MODELS list: 23
Eligible models spanning 1981-2010: 12
['CanSIPS-IC3', 'NASA-GMAO-062012', 'NCAR-CESM1', 'CanSIPSv2', 'GFDL-CM2p5-FLOR-A06', 'CMC1-CanCM3', 'GFDL-CM2p5-FLOR-B01', 'CMC2-CanCM4', 'GEM-NEMO', 'NASA-GMAO', 'CanCM4i', 'NASA-GEOSS2S']

Excluded models without the complete climatology window:
  GFDL-CM2p1: available 1982-2012
  CanSIPS-IC4: available 1990-2024
  GFDL-CM2p1-aer04: available 1982-2021
  NCEP-CFSv1: available 1981-2009
  NCEP-CFSv2: available 1982-2010
  GFDL-SPEAR: availa

## Run preprocessing

This can take a while on the first run because it reads each member file and writes per-model cache files under `OUTDIR/sst_index/processed/`. Later runs reuse those cached files unless `FORCE = True`. The final NMME index time series are written under `OUTDIR/sst_index/timeseries/`.


In [4]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--nmme-root", str(NMME_ROOT),
    "--nmme-fixed-dir", str(NMME_FIXED_DIR),
    "--outdir", str(OUTDIR),
    "--data-start", str(DATA_START_YEAR),
    "--data-end", str(DATA_END_YEAR),
    "--clim-start", str(CLIM_START),
    "--clim-end", str(CLIM_END),
    "--regions", *(str(region) for region in REGIONS),
]

if CUSTOM_REGIONS:
    cmd.extend(["--custom-regions", json.dumps(CUSTOM_REGIONS)])

# Pass the resolved list explicitly so the command records the exact model selection.
cmd.extend(["--models", *SELECTED_MODELS])
cmd.append("--nmme-sst-land-mask" if NMME_SST_LAND_MASK else "--no-nmme-sst-land-mask")

if FORCE:
    cmd.append("--force")

env = os.environ.copy()
conda_prefix = Path(sys.prefix)
for env_name, relpath in {
    "GDAL_DATA": "share/gdal",
    "PROJ_LIB": "share/proj",
    "PROJ_DATA": "share/proj",
}.items():
    candidate = conda_prefix / relpath
    if candidate.exists():
        env[env_name] = str(candidate)

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)


Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python /global/u2/z/zhan391/code/ESP-Lab/scripts/run_nmme_nino34_yeager_diag.py --nmme-root /global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member --nmme-fixed-dir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/fixed --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME --data-start 1980 --data-end 2020 --clim-start 1981 --clim-end 2010 --regions IOD TNI ONI RONI Nino12 Nino3 Nino3.4 Nino4 TNA TSA PACWRAMPOOL AtlNino AtlMDR ELI --custom-regions {"Nino12": {"lonlat": [270.0, 280.0, -10.0, 0.0], "long_name": "Nino 1+2 regional mean SST"}, "Nino3": {"lonlat": [210.0, 270.0, -5.0, 5.0], "long_name": "Nino 3 regional mean SST"}, "Nino3.4": {"lonlat": [190.0, 240.0, -5.0, 5.0], "long_name": "Nino 3.4 regional mean SST"}, "Nino4": {"lonlat": [160.0, 210.0, -5.0, 5.0], "long_name": "Nino 4 regional mean SST"}, "TNA": {"lonlat": [305.0, 345.0, 5.0, 25.0], "long_name": "TNA regional mean SST"}, "TSA": {"lonlat": [330.0, 10.0, -2

/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python

[NMME] data window: 1980-2020
[NMME] skill climatology window: 1981-2010
[NMME] processing 12 model(s) from member-split archive
[NMME] processing region: IOD
Saved NMME SST-index time series under: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries
[NMME] processing region: TNI
Saved NMME SST-index time series under: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries
[NMME] processing region: ONI
Saved NMME SST-index time series under: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries
[NMME] processing region: RONI
Saved NMME SST-index time series under: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries
[NMME] processing region: Nino12
Saved NMME SST-index time series under: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries
[NMME] processing region: Nino3
Saved NMME SST-index time series under: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries
[NMME] processing region: Nino3.4
Saved NMME SST-index time s

CompletedProcess(args=['/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python', '/global/u2/z/zhan391/code/ESP-Lab/scripts/run_nmme_nino34_yeager_diag.py', '--nmme-root', '/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member', '--nmme-fixed-dir', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/fixed', '--outdir', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME', '--data-start', '1980', '--data-end', '2020', '--clim-start', '1981', '--clim-end', '2010', '--regions', 'IOD', 'TNI', 'ONI', 'RONI', 'Nino12', 'Nino3', 'Nino3.4', 'Nino4', 'TNA', 'TSA', 'PACWRAMPOOL', 'AtlNino', 'AtlMDR', 'ELI', '--custom-regions', '{"Nino12": {"lonlat": [270.0, 280.0, -10.0, 0.0], "long_name": "Nino 1+2 regional mean SST"}, "Nino3": {"lonlat": [210.0, 270.0, -5.0, 5.0], "long_name": "Nino 3 regional mean SST"}, "Nino3.4": {"lonlat": [190.0, 240.0, -5.0, 5.0], "long_name": "Nino 3.4 regional mean SST"}, "Nino4": {"lonlat": [160.0, 210.0, -5.0, 5.0], "long_name": "Nino 4 regional mean SST"}, "TNA": {"lonlat

## Inspect generated time-series file


In [5]:
missing_timeseries_files = [path for path in EXPECTED_TIMESERIES_FILES.values() if not path.is_file()]
if missing_timeseries_files:
    raise FileNotFoundError("Expected time-series files were not written:\n" + "\n".join(str(path) for path in missing_timeseries_files[:50]))

ts_ds = xr.open_dataset(EXPECTED_NINO34_TIMESERIES_FILE)
print(EXPECTED_NINO34_TIMESERIES_FILE)
print(ts_ds)
print("models:")
print(ts_ds.model.values.tolist())


/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/sst_index/timeseries/NMME05_Nino34SST_seas_dd_1980_2020.nc
<xarray.Dataset> Size: 161kB
Dimensions:  (model: 12, M: 20, Y: 41, L: 4)
Coordinates:
  * model    (model) <U19 912B 'CanSIPS-IC3' ... 'NASA-GEOSS2S'
  * M        (M) int64 160B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
  * Y        (Y) int64 328B 1980 1981 1982 1983 1984 ... 2017 2018 2019 2020
    month    (Y) int64 328B ...
  * L        (L) int64 32B 3 6 9 12
Data variables:
    sst      (model, M, Y, L) float32 157kB ...
    time     (Y, L) object 1kB ...
Attributes:
    region:                  Nino3.4
    climatology_start_year:  1981
    climatology_end_year:    2010
models:
['CanSIPS-IC3', 'NASA-GMAO-062012', 'NCAR-CESM1', 'CanSIPSv2', 'GFDL-CM2p5-FLOR-A06', 'CMC1-CanCM3', 'GFDL-CM2p5-FLOR-B01', 'CMC2-CanCM4', 'GEM-NEMO', 'NASA-GMAO', 'CanCM4i', 'NASA-GEOSS2S']
